<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 1 · Introduction

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/01_dxquant_introduction.ipynb)


## Notebook Goals

This notebook introduces the core building blocks of `dxquant` on a minimal example. You will

- create a `MarketEnvironment` and attach a discount curve,
- specify a stochastic process (Geometric Brownian Motion),
- price a European call option by Monte Carlo simulation, and
- compare the Monte Carlo result to an analytical benchmark.

The examples below are deliberately small so you can reuse them as a template for later notebooks.

Sampling errors use independent paths or antithetic pair means. A single moment-matched batch reports `NaN` for its error; use independent seeded replications to measure batch dispersion. A shared `scenario_set` explicitly asserts ordered joint scenarios for portfolio payoff risk; it does not construct correlations.


## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt
from pprint import pprint

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

## Market Environment and Discounting

The `MarketEnvironment` groups a pricing date with market inputs such as curves and constants. For
this first notebook, a flat continuously compounded short rate is sufficient.

In [ ]:
pricing_date = dt.date(2024, 1, 1)
rate = 0.05

env = dx.MarketEnvironment(name="bsm", pricing_date=pricing_date)
env.add_curve("discount", dx.ConstantShortRateCurve("disc", pricing_date, rate))

pprint(env.snapshot())

## Process and Monte Carlo Pricer

We assume a single equity-style risk factor following Geometric Brownian Motion (GBM). A fixed
seed makes the Monte Carlo estimate deterministic for tutorial purposes.

In [ ]:
spot = 100.0
strike = 105.0
vol = 0.20
maturity = dt.date(2025, 1, 1)

process = dx.GeometricBrownianMotion(
    drift=rate,
    volatility=vol,
    seed=123,
    moment_matching=False,
)
pricer = dx.MonteCarloPricer(
    env=env,
    process=process,
    discount_curve_key="discount",
    paths=140000,
    steps=96,
)

process

## Price a European Call and Cross-Check

The Monte Carlo pricer returns pricing statistics (price, standard error, discounted payoffs).
For a GBM under a constant short rate, an analytical Black–Scholes benchmark is available.

In [ ]:
payoff = dx.EuropeanCallPayoff(strike=strike)
valuation = dx.EuropeanOptionValuation(
    pricer=pricer,
    payoff=payoff,
    maturity=maturity,
    spot=spot,
)

mc_result = pricer.price_terminal(payoff, maturity=maturity, spot=spot)
analytical = valuation.analytical_price()

print(f"Monte Carlo price:   {mc_result.price:.6f}")
print(f"Std. error:          {mc_result.std_error:.6f}")
print(f"Analytical price:    {float(analytical):.6f}")
print(f"Abs. difference:     {abs(mc_result.price - float(analytical)):.6f}")

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">